In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# %% 셀 0: 데이터 로드
import os, json, random, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import polars as pl
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

POS_DIR = "./data/8_telop_position"
STT_DIR = "./data/4_stt_results"
EMB_PATH = "./data/8_text_embeddings.pt"
GRID_W = 80
GRID_H = 80
EVAL_PER_CHANNEL = 5
SEED = 42
NUM_WORKERS = 32
FPS = 10
MAX_ACTIVE = 150
MAX_TEXT_LEN = 200
SCALAR_DIM = 8
SEG_SCALAR_DIM = 4

text2emb = torch.load(EMB_PATH, weights_only=True)
EMB_DIM = next(iter(text2emb.values())).shape[0]
ZERO_EMB = torch.zeros(EMB_DIM)
print(f"✅ 임베딩: {len(text2emb):,}개 dim={EMB_DIM}")


def stt_frames_to_segments(df_stt):
    rows = df_stt.sort("frame_num").to_dicts()
    segments = []
    cur_text = None
    cur_start_frame = None
    prev_frame = None
    for r in rows:
        t = r["stt_text"]
        f = int(r["frame_num"])
        if t != cur_text:
            if cur_text is not None and cur_text != "":
                segments.append({
                    "start": cur_start_frame / FPS,
                    "end": (prev_frame + 1) / FPS,
                    "text": cur_text.strip(),
                })
            cur_text = t
            cur_start_frame = f
        prev_frame = f
    if cur_text is not None and cur_text != "":
        segments.append({
            "start": cur_start_frame / FPS,
            "end": (prev_frame + 1) / FPS,
            "text": cur_text.strip(),
        })
    return segments


def load_one(args):
    channel, path = args
    with open(path, "r") as f:
        data = json.load(f)
    instances = data.get("instances", [])
    duration = data.get("duration", 0.1)
    if instances:
        duration = max(duration, max(inst["end_sec"] for inst in instances))
    video_name = data.get("video", "")
    file_id = os.path.basename(path)[:-5]
    inst_list = []
    for inst in instances:
        gx = int(np.clip(round(inst["grid_x"]), 0, GRID_W - 1))
        gy = int(np.clip(round(inst["grid_y"]), 0, GRID_H - 1))
        gw = int(np.clip(round(inst["grid_w"]), 1, GRID_W))
        gh = int(np.clip(round(inst["grid_h"]), 1, GRID_H))
        inst_list.append({
            "text": inst["text"],
            "text_len": len(inst["text"]),
            "start": inst["start_sec"],
            "end": inst["end_sec"],
            "x": gx, "y": gy, "w": gw, "h": gh,
        })
    stt_path = os.path.join(STT_DIR, channel, file_id + ".parquet")
    stt_segments = []
    if os.path.exists(stt_path):
        try:
            df_stt = pl.read_parquet(stt_path, glob=False)
            stt_segments = stt_frames_to_segments(df_stt)
        except:
            pass
    return {
        "channel": channel, "video_name": video_name, "file_id": file_id,
        "instances": inst_list, "stt_segments": stt_segments, "duration": duration,
    }


json_paths = []
for channel in sorted(os.listdir(POS_DIR)):
    ch_dir = os.path.join(POS_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    for fname in sorted(os.listdir(ch_dir)):
        if fname.endswith(".json"):
            json_paths.append((channel, os.path.join(ch_dir, fname)))

samples = []
channel_set = set()
with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(load_one, args): args for args in json_paths}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="로드"):
        result = fut.result()
        channel_set.add(result["channel"])
        samples.append(result)

channels = sorted(channel_set)
channel2id = {ch: i for i, ch in enumerate(channels)}

rng = random.Random(SEED)
by_channel = {}
for s in samples:
    by_channel.setdefault(s["channel"], []).append(s)

train_samples = []
eval_samples = []
for ch, ch_samples in by_channel.items():
    ch_samples.sort(key=lambda s: s["file_id"])
    rng.shuffle(ch_samples)
    n_eval = min(EVAL_PER_CHANNEL, len(ch_samples))
    eval_samples.extend(ch_samples[:n_eval])
    train_samples.extend(ch_samples[n_eval:])

train_samples = [s for s in train_samples if len(s["instances"]) > 0]
eval_samples  = [s for s in eval_samples  if len(s["instances"]) > 0]

print(f"✅ 채널: {len(channels)}개  train: {len(train_samples):,}  eval: {len(eval_samples):,}")


✅ 임베딩: 2,167,019개 dim=1024


로드: 100%|██████████| 66400/66400 [00:13<00:00, 4814.38it/s]


✅ 채널: 664개  train: 56,892  eval: 2,984


In [2]:
# %% 셀 1: 채널 서브샘플링
rng_ch = random.Random(42)
sampled_channels = rng_ch.sample(channels, 67)
sampled_set = set(sampled_channels)

train_samples = [s for s in train_samples if s["channel"] in sampled_set]
eval_samples  = [s for s in eval_samples  if s["channel"] in sampled_set]
channels = sampled_channels
channel2id = {ch: i for i, ch in enumerate(channels)}

print(f"🔬 {len(channels)}개 채널  train: {len(train_samples):,}  eval: {len(eval_samples):,}")


🔬 67개 채널  train: 5,574  eval: 288


In [3]:
# %% 셀 2: Segment-based Dataset
from torch.utils.data import Dataset, DataLoader

BATCH_SIZE = 512
NUM_WORKERS_DL = 8
MAX_SEG_LOG = math.log(MAX_ACTIVE * 100)


def make_segments(s):
    insts = s["instances"]
    duration = max(s["duration"], 0.1)
    bounds = {0.0, duration}
    for inst in insts:
        bounds.add(inst["start"])
        bounds.add(inst["end"])
    bounds = sorted(b for b in bounds if 0 <= b <= duration)
    segments = []
    for t0, t1 in zip(bounds, bounds[1:]):
        if t1 - t0 <= 1e-4:
            continue
        mid = (t0 + t1) / 2
        active_idx = [j for j, inst in enumerate(insts)
                      if inst["start"] <= mid < inst["end"]]
        if not active_idx:
            continue
        if len(active_idx) > MAX_ACTIVE:
            active_idx = sorted(active_idx,
                                key=lambda j: -insts[j]["text_len"])[:MAX_ACTIVE]
        segments.append({
            "t0": t0, "t1": t1, "mid": mid,
            "active_idx": active_idx,
            "seg_len_frames": max(1.0, (t1 - t0) * FPS),
        })
    return segments


def build_segment_index(video_samples):
    entries = []
    for s in tqdm(video_samples, desc="segment 분해"):
        for seg in make_segments(s):
            entries.append((s, seg))
    return entries


train_entries = build_segment_index(train_samples)
eval_entries  = build_segment_index(eval_samples)
print(f"📦 train: {len(train_entries):,}  eval: {len(eval_entries):,} segments")


class SegmentDataset(Dataset):
    def __init__(self, entries, channel2id, text2emb):
        self.entries = entries
        self.channel2id = channel2id
        self.text2emb = text2emb

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        s, seg = self.entries[idx]
        insts = s["instances"]
        active_idx = seg["active_idx"]
        N = len(active_idx)
        duration = max(s["duration"], 0.1)
        mid = seg["mid"]

        channel_emb = F.normalize(self.text2emb.get(s["channel"],    ZERO_EMB), dim=-1)
        video_emb   = F.normalize(self.text2emb.get(s["video_name"], ZERO_EMB), dim=-1)

        stt_emb = ZERO_EMB; has_stt = 0.0
        for sg in s["stt_segments"]:
            if sg["start"] <= mid < sg["end"]:
                stt_emb = F.normalize(self.text2emb.get(sg["text"], ZERO_EMB), dim=-1)
                has_stt = 1.0
                break

        active_insts = [insts[j] for j in active_idx]
        inst_embs = torch.stack([
            F.normalize(self.text2emb.get(inst["text"], ZERO_EMB), dim=-1)
            for inst in active_insts
        ])
        diff_ch  = inst_embs - channel_emb.unsqueeze(0)
        diff_vid = inst_embs - video_emb.unsqueeze(0)
        diff_stt = inst_embs - stt_emb.unsqueeze(0)

        ch_sims  = F.cosine_similarity(inst_embs, channel_emb.unsqueeze(0), dim=-1).numpy()
        vid_sims = F.cosine_similarity(inst_embs, video_emb.unsqueeze(0),   dim=-1).numpy()
        stt_sims = F.cosine_similarity(inst_embs, stt_emb.unsqueeze(0),     dim=-1).numpy()

        inst_scalars = np.zeros((N, SCALAR_DIM), dtype=np.float32)
        inst_scalars[:, 0] = [inst["text_len"] / MAX_TEXT_LEN for inst in active_insts]
        inst_scalars[:, 1] = ch_sims
        inst_scalars[:, 2] = vid_sims
        inst_scalars[:, 3] = stt_sims
        inst_scalars[:, 4] = has_stt
        inst_scalars[:, 5] = [inst["start"] / duration for inst in active_insts]
        inst_scalars[:, 6] = [inst["end"]   / duration for inst in active_insts]
        inst_scalars[:, 7] = [(inst["end"] - inst["start"]) / duration for inst in active_insts]

        seg_scalar = np.array([
            mid / duration,
            (seg["t1"] - seg["t0"]) / duration,
            min(N / MAX_ACTIVE, 1.0),
            np.log1p(seg["seg_len_frames"]) / MAX_SEG_LOG,
        ], dtype=np.float32)

        # GT bbox 정규화 [-1, 1]
        # cx, cy, w, h → 2*x/GRID - 1
        gt_norm = np.zeros((N, 4), dtype=np.float32)
        for i, inst in enumerate(active_insts):
            cx, cy, w, h = inst["x"], inst["y"], inst["w"], inst["h"]
            gt_norm[i, 0] = 2.0 * cx / GRID_W - 1.0
            gt_norm[i, 1] = 2.0 * cy / GRID_H - 1.0
            gt_norm[i, 2] = 2.0 * w  / GRID_W - 1.0
            gt_norm[i, 3] = 2.0 * h  / GRID_H - 1.0

        return {
            "channel_id":   self.channel2id[s["channel"]],
            "inst_embs":    inst_embs,
            "diff_ch":      diff_ch,
            "diff_vid":     diff_vid,
            "diff_stt":     diff_stt,
            "inst_scalars": torch.from_numpy(inst_scalars),
            "seg_scalar":   torch.from_numpy(seg_scalar),
            "channel_emb":  channel_emb,
            "video_emb":    video_emb,
            "stt_emb":      stt_emb,
            "gt_norm":      torch.from_numpy(gt_norm),
            "seg_len":      float(seg["seg_len_frames"]),
            "n_active":     N,
        }


def collate(batch):
    B = len(batch)
    max_n = max(b["n_active"] for b in batch)
    D = batch[0]["inst_embs"].shape[1]

    inst_embs = torch.zeros(B, max_n, D)
    diff_ch   = torch.zeros(B, max_n, D)
    diff_vid  = torch.zeros(B, max_n, D)
    diff_stt  = torch.zeros(B, max_n, D)
    inst_scalars = torch.zeros(B, max_n, SCALAR_DIM)
    gt_norm   = torch.zeros(B, max_n, 4)
    inst_mask = torch.zeros(B, max_n, dtype=torch.bool)

    seg_scalar = torch.zeros(B, SEG_SCALAR_DIM)
    channel_emb = torch.zeros(B, D)
    video_emb   = torch.zeros(B, D)
    stt_emb     = torch.zeros(B, D)
    channel_id  = torch.zeros(B, dtype=torch.long)
    seg_len     = torch.zeros(B)

    for bi, b in enumerate(batch):
        n = b["n_active"]
        inst_embs[bi, :n]    = b["inst_embs"]
        diff_ch[bi, :n]      = b["diff_ch"]
        diff_vid[bi, :n]     = b["diff_vid"]
        diff_stt[bi, :n]     = b["diff_stt"]
        inst_scalars[bi, :n] = b["inst_scalars"]
        gt_norm[bi, :n]      = b["gt_norm"]
        inst_mask[bi, :n]    = True
        seg_scalar[bi]   = b["seg_scalar"]
        channel_emb[bi]  = b["channel_emb"]
        video_emb[bi]    = b["video_emb"]
        stt_emb[bi]      = b["stt_emb"]
        channel_id[bi]   = b["channel_id"]
        seg_len[bi]      = b["seg_len"]

    return {
        "inst_embs":    inst_embs,
        "diff_ch":      diff_ch,
        "diff_vid":     diff_vid,
        "diff_stt":     diff_stt,
        "inst_scalars": inst_scalars,
        "gt_norm":      gt_norm,
        "inst_mask":    inst_mask,
        "seg_scalar":   seg_scalar,
        "channel_emb":  channel_emb,
        "video_emb":    video_emb,
        "stt_emb":      stt_emb,
        "channel_id":   channel_id,
        "seg_len":      seg_len,
    }


train_ds = SegmentDataset(train_entries, channel2id, text2emb)
eval_ds  = SegmentDataset(eval_entries,  channel2id, text2emb)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS_DL, collate_fn=collate,
                          pin_memory=True, drop_last=True)
eval_loader  = DataLoader(eval_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS_DL, collate_fn=collate,
                          pin_memory=True)
print(f"🚚 train batches: {len(train_loader):,}  eval batches: {len(eval_loader):,}")


segment 분해: 100%|██████████| 288/288 [00:00<00:00, 2892.98it/s]

📦 train: 307,294  eval: 15,451 segments
🚚 train batches: 600  eval batches: 31


In [4]:
# %% 셀 3: LayoutFlow Model (transformer + velocity head)
D_MODEL = EMB_DIM
N_HEADS = 8
N_LAYERS = 6
FFN_DIM = D_MODEL * 4
DROPOUT = 0.1
N_CHANNELS = len(channels)
T_EMB_DIM = 128            # time embedding dim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def time_embedding(t, dim):
    """t: [...] in [0, 1]. Returns [..., dim] sinusoidal embedding."""
    half = dim // 2
    freqs = torch.exp(-math.log(10000.0) *
                      torch.arange(half, device=t.device, dtype=torch.float32) / half)
    args = t.unsqueeze(-1).float() * freqs * 2 * math.pi
    return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class FlowModel(nn.Module):
    def __init__(self, emb_dim, n_channels, d_model=D_MODEL, n_heads=N_HEADS,
                 n_layers=N_LAYERS, ffn=FFN_DIM, dropout=DROPOUT, t_dim=T_EMB_DIM):
        super().__init__()
        self.channel_embed = nn.Embedding(n_channels, d_model)

        # context: channel / video / STT / seg scalar
        self.ctx_ch_proj     = nn.Linear(emb_dim, d_model)
        self.ctx_vid_proj    = nn.Linear(emb_dim, d_model)
        self.ctx_stt_proj    = nn.Linear(emb_dim, d_model)
        self.ctx_scalar_proj = nn.Linear(SEG_SCALAR_DIM, d_model)

        # instance: diff vectors + scalars
        self.diff_ch_proj     = nn.Linear(emb_dim, d_model)
        self.diff_vid_proj    = nn.Linear(emb_dim, d_model)
        self.diff_stt_proj    = nn.Linear(emb_dim, d_model)
        self.inst_scalar_proj = nn.Linear(SCALAR_DIM, d_model)

        # x_t (current noisy bbox) projection
        self.x_proj = nn.Linear(4, d_model)

        # time embedding projection
        self.t_proj = nn.Sequential(
            nn.Linear(t_dim, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

        self.type_embed = nn.Embedding(2, d_model)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=ffn,
            dropout=dropout, activation="gelu", batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)

        # velocity head: D → 4 (velocity for cx, cy, w, h)
        self.v_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, 4),
        )
        nn.init.zeros_(self.v_head[-1].weight)
        nn.init.zeros_(self.v_head[-1].bias)
        self.t_dim = t_dim

    def forward(self, batch, x_t, t):
        """
        batch: condition dict
        x_t:   [B, N, 4] current noisy bbox
        t:     [B] or scalar in [0, 1]
        Returns: v_pred [B, N, 4]
        """
        B = batch["inst_embs"].size(0)
        N = batch["inst_embs"].size(1)

        # time embedding
        if t.dim() == 0:
            t = t.expand(B)
        t_emb = time_embedding(t, self.t_dim)               # [B, t_dim]
        t_tok = self.t_proj(t_emb)                          # [B, d_model]

        # instance token
        inst_tok = (self.diff_ch_proj(batch["diff_ch"])
                    + self.diff_vid_proj(batch["diff_vid"])
                    + self.diff_stt_proj(batch["diff_stt"])
                    + self.inst_scalar_proj(batch["inst_scalars"])
                    + self.x_proj(x_t)                       # current x_t
                    + t_tok.unsqueeze(1))                    # time broadcast
        inst_tok = inst_tok + self.type_embed.weight[1]

        # context token
        ch_id_emb = self.channel_embed(batch["channel_id"])
        ctx_tok = (self.ctx_ch_proj(batch["channel_emb"])
                   + self.ctx_vid_proj(batch["video_emb"])
                   + self.ctx_stt_proj(batch["stt_emb"])
                   + self.ctx_scalar_proj(batch["seg_scalar"])
                   + ch_id_emb
                   + t_tok)                                  # time도 context에
        ctx_tok = ctx_tok + self.type_embed.weight[0]
        ctx_tok = ctx_tok.unsqueeze(1)

        tokens = torch.cat([ctx_tok, inst_tok], dim=1)
        ctx_mask = torch.zeros(B, 1, dtype=torch.bool, device=tokens.device)
        pad_mask = torch.cat([ctx_mask, ~batch["inst_mask"]], dim=1)

        out = self.encoder(tokens, src_key_padding_mask=pad_mask)
        inst_out = out[:, 1:]                                # [B, N, D]
        v_pred = self.v_head(inst_out)                       # [B, N, 4]
        return v_pred


model = FlowModel(EMB_DIM, N_CHANNELS).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"🧠 모델 params: {n_params/1e6:.2f}M | D={D_MODEL} L={N_LAYERS} H={N_HEADS}")
print(f"   LayoutFlow: condition + x_t + t → velocity (linear rectified flow)")


/tmp/ipykernel_964827/2027726829.py:57: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)


🧠 모델 params: 84.20M | D=1024 L=6 H=8
   LayoutFlow: condition + x_t + t → velocity (linear rectified flow)


In [ ]:
# %% 셀 4: 학습 (MSE on velocity)
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS = 50
LR = 1e-4
WEIGHT_DECAY = 0.01
N_INFER_STEPS = 8                   # ODE step 수
SAVE_DIR = "./model/8_layout_flow"
os.makedirs(SAVE_DIR, exist_ok=True)


def flow_loss_per_token(v_pred, v_true, inst_mask):
    """[B, N, 4] MSE per token → [B, N]."""
    diff = (v_pred.float() - v_true.float()) ** 2
    return diff.mean(-1)                          # [B, N]


def aggregate_per_token(loss_tok, inst_mask, seg_len_w):
    if not inst_mask.any():
        return loss_tok.sum() * 0.0
    m = inst_mask.float()
    counts = m.sum(dim=1).clamp(min=1.0)
    seg_loss = (loss_tok * m).sum(dim=1) / counts
    w_n = seg_len_w / seg_len_w.sum().clamp(min=1e-8)
    return (seg_loss * w_n).sum()


def batch_to_device(batch, device):
    return {k: (v.to(device) if isinstance(v, torch.Tensor) else v)
            for k, v in batch.items()}


@torch.no_grad()
def flow_inference(model, batch, n_steps=N_INFER_STEPS):
    """Euler ODE integration. noise → target."""
    B, N = batch["inst_embs"].shape[:2]
    x_t = torch.randn(B, N, 4, device=device)         # x_0 = noise ~ N(0,1)
    dt = 1.0 / n_steps
    for i in range(n_steps):
        t = torch.full((B,), i * dt, device=device)
        v_pred = model(batch, x_t, t)
        x_t = x_t + dt * v_pred
    # x_t ≈ target (in [-1, 1] space)
    return x_t


def denormalize(x):
    """x: [B, N, 4] in [-1, 1] → (cx, cy, w, h) in pixel."""
    cx = ((x[..., 0] + 1) / 2 * GRID_W).clamp(0, GRID_W - 1)
    cy = ((x[..., 1] + 1) / 2 * GRID_H).clamp(0, GRID_H - 1)
    w  = ((x[..., 2] + 1) / 2 * GRID_W).clamp(1, GRID_W)
    h  = ((x[..., 3] + 1) / 2 * GRID_H).clamp(1, GRID_H)
    return torch.stack([cx, cy, w, h], dim=-1)


def gt_to_bbox(gt_norm):
    """gt_norm: [B,N,4] in [-1,1] (cx,cy,w,h) → bbox (x0,y0,x1,y1) in pixel."""
    params = denormalize(gt_norm)
    cx, cy, w, h = params.unbind(-1)
    x0 = cx - w / 2; y0 = cy - h / 2
    x1 = cx + w / 2; y1 = cy + h / 2
    return torch.stack([x0, y0, x1, y1], dim=-1), params


@torch.no_grad()
def compute_metrics(pred_params, gt_norm, inst_mask, seg_len_w):
    if not inst_mask.any():
        return None
    cx, cy, w, h = pred_params.float().unbind(-1)
    gt_params = denormalize(gt_norm).float()
    gt_cx, gt_cy, gt_w, gt_h = gt_params.unbind(-1)

    px0 = cx - w/2; py0 = cy - h/2
    px1 = cx + w/2; py1 = cy + h/2
    gx0 = gt_cx - gt_w/2; gy0 = gt_cy - gt_h/2
    gx1 = gt_cx + gt_w/2; gy1 = gt_cy + gt_h/2

    ix0 = torch.max(px0, gx0); iy0 = torch.max(py0, gy0)
    ix1 = torch.min(px1, gx1); iy1 = torch.min(py1, gy1)
    inter = (ix1 - ix0).clamp(min=0) * (iy1 - iy0).clamp(min=0)
    pa = (px1 - px0).clamp(min=0) * (py1 - py0).clamp(min=0)
    ga = (gx1 - gx0).clamp(min=0) * (gy1 - gy0).clamp(min=0)
    iou = inter / (pa + ga - inter + 1e-8)

    m = inst_mask.float()
    counts = m.sum(dim=1).clamp(min=1.0)
    w_n = seg_len_w / seg_len_w.sum().clamp(min=1e-8)
    miou   = ((iou * m).sum(1) / counts * w_n).sum().item()
    iou_50 = (((iou > 0.5).float() * m).sum(1) / counts * w_n).sum().item()
    iou_75 = (((iou > 0.75).float() * m).sum(1) / counts * w_n).sum().item()
    err_cx = (((cx - gt_cx).abs() * m).sum(1) / counts * w_n).sum().item()
    err_cy = (((cy - gt_cy).abs() * m).sum(1) / counts * w_n).sum().item()
    err_w  = (((w  - gt_w).abs()  * m).sum(1) / counts * w_n).sum().item()
    err_h  = (((h  - gt_h).abs()  * m).sum(1) / counts * w_n).sum().item()
    return {
        "mIoU": miou, "IoU@0.5": iou_50, "IoU@0.75": iou_75,
        "errCX": err_cx, "errCY": err_cy, "errW": err_w, "errH": err_h,
    }


optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
best_score = -1.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0; n_b = 0
    for batch in tqdm(train_loader, desc=f"[{epoch}/{EPOCHS}] train", leave=False):
        batch = batch_to_device(batch, device)
        target = batch["gt_norm"]                    # [B, N, 4] in [-1, 1]
        B, N = target.shape[:2]

        # sample t ~ Uniform(0, 1) per sample
        t = torch.rand(B, device=device)
        # sample noise ~ N(0, 1)
        noise = torch.randn_like(target)
        # linear interpolation
        x_t = (1 - t).view(B, 1, 1) * noise + t.view(B, 1, 1) * target
        v_true = target - noise                       # constant velocity

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            v_pred = model(batch, x_t, t)
            loss_tok = flow_loss_per_token(v_pred, v_true, batch["inst_mask"])
            loss = aggregate_per_token(loss_tok, batch["inst_mask"], batch["seg_len"])

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
        n_b += 1

    model.eval()
    eval_loss = 0.0; n_eb = 0
    all_metrics = []
    with torch.no_grad():
        for batch in eval_loader:
            batch = batch_to_device(batch, device)
            target = batch["gt_norm"]
            B, N = target.shape[:2]
            t = torch.rand(B, device=device)
            noise = torch.randn_like(target)
            x_t = (1 - t).view(B, 1, 1) * noise + t.view(B, 1, 1) * target
            v_true = target - noise
            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
                v_pred = model(batch, x_t, t)
                loss_tok = flow_loss_per_token(v_pred, v_true, batch["inst_mask"])
                eloss = aggregate_per_token(loss_tok, batch["inst_mask"], batch["seg_len"])
            eval_loss += eloss.item(); n_eb += 1

            # ODE inference for mIoU
            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
                x_final = flow_inference(model, batch, n_steps=N_INFER_STEPS)
            pred_params = denormalize(x_final)
            m = compute_metrics(pred_params, batch["gt_norm"],
                                batch["inst_mask"], batch["seg_len"])
            if m is not None:
                all_metrics.append((m, batch["seg_len"].sum().item()))

    tw = sum(w for _, w in all_metrics) or 1.0
    agg = {k: sum(m[k] * w for m, w in all_metrics) / tw for k in all_metrics[0][0]}

    scheduler.step()
    msg = ("[{ep}/{ne}]  train={tl:.4f}  eval(velocity)={el:.4f}  "
           "mIoU={miou:.4f}  IoU@0.5={i50:.3f}  IoU@0.75={i75:.3f}  "
           "errCX={ecx:.2f}  errCY={ecy:.2f}  errW={ew:.2f}  errH={eh:.2f}  lr={lr:.2e}"
           ).format(ep=epoch, ne=EPOCHS,
                    tl=train_loss / max(n_b, 1),
                    el=eval_loss / max(n_eb, 1),
                    miou=agg["mIoU"], i50=agg["IoU@0.5"], i75=agg["IoU@0.75"],
                    ecx=agg["errCX"], ecy=agg["errCY"],
                    ew=agg["errW"], eh=agg["errH"],
                    lr=scheduler.get_last_lr()[0])
    print(msg)

    if agg["mIoU"] > best_score:
        best_score = agg["mIoU"]
        torch.save({"model": model.state_dict(), "epoch": epoch,
                    "mIoU": best_score},
                   os.path.join(SAVE_DIR, "best.pt"))
        print(f"   💾 best 갱신 (mIoU={best_score:.4f})")


[1/50]  train=0.5302  eval(velocity)=0.4346  mIoU=0.0465  IoU@0.5=0.021  IoU@0.75=0.001  errCX=11.86  errCY=18.57  errW=17.54  errH=1.92  lr=9.99e-05
   💾 best 갱신 (mIoU=0.0465)


[2/50]  train=0.4422  eval(velocity)=0.4605  mIoU=0.0300  IoU@0.5=0.010  IoU@0.75=0.000  errCX=10.32  errCY=20.13  errW=20.00  errH=2.04  lr=9.96e-05


[3/50]  train=0.4151  eval(velocity)=0.3824  mIoU=0.0798  IoU@0.5=0.050  IoU@0.75=0.004  errCX=9.08  errCY=17.38  errW=16.18  errH=1.56  lr=9.91e-05
   💾 best 갱신 (mIoU=0.0798)


[4/50]  train=0.4005  eval(velocity)=0.3814  mIoU=0.0848  IoU@0.5=0.043  IoU@0.75=0.004  errCX=7.89  errCY=17.46  errW=16.03  errH=1.97  lr=9.84e-05
   💾 best 갱신 (mIoU=0.0848)


[5/50]  train=0.3896  eval(velocity)=0.3706  mIoU=0.0944  IoU@0.5=0.052  IoU@0.75=0.002  errCX=8.17  errCY=17.64  errW=15.87  errH=2.09  lr=9.76e-05
   💾 best 갱신 (mIoU=0.0944)


...

[24/50]  train=0.2571  eval(velocity)=0.2528  mIoU=0.1660  IoU@0.5=0.133  IoU@0.75=0.033  errCX=6.43  errCY=12.19  errW=9.96  errH=1.13  lr=5.31e-05
   💾 best 갱신 (mIoU=0.1660)


[25/50]  train=0.2584  eval(velocity)=0.2679  mIoU=0.1733  IoU@0.5=0.139  IoU@0.75=0.028  errCX=5.92  errCY=11.31  errW=10.08  errH=1.12  lr=5.00e-05
   💾 best 갱신 (mIoU=0.1733)


[26/50]  train=0.2472  eval(velocity)=0.2567  mIoU=0.1568  IoU@0.5=0.123  IoU@0.75=0.021  errCX=6.29  errCY=11.59  errW=10.32  errH=1.13  lr=4.69e-05


[27/50]  train=0.2477  eval(velocity)=0.2592  mIoU=0.1683  IoU@0.5=0.148  IoU@0.75=0.034  errCX=5.91  errCY=11.30  errW=9.71  errH=1.09  lr=4.37e-05


[28/50] train:   0%|          | 0/600 [00:00<?, ?it/s]